In [ ]:
import os
import numpy as np
import torch
from torch.utils.data import DataLoader
import torchvision
from torchvision import transforms
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from sklearn.decomposition import PCA
from scipy.stats import gaussian_kde
from IPython.display import HTML
from model.resnet11 import MnistFM

## Config

In [ ]:
mnist_data_dir = "/workspace/data"
batch_size = 256
model_path = "/workspace/samples/sample_model/resnet11_epoch100.pth"
movie_save_dir = "/workspace/movie"

os.makedirs(movie_save_dir, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## DataLoader

In [ ]:
mnist_dataset = torchvision.datasets.MNIST(
    root=mnist_data_dir, 
    train=True, 
    transform=transforms.ToTensor(), 
    download=True
)

dataloader = DataLoader(
    mnist_dataset,
    batch_size=batch_size,
    shuffle=False
)

## MNIST画像 PCA(2次元圧縮)

In [ ]:
all_data = []
all_labels = []

for imgs, labels in dataloader:
    # imgs: (batch, 1, 28, 28) -> (batch, 784)
    all_data.append(imgs.view(imgs.size(0), -1).numpy())
    all_labels.append(labels.numpy())

all_data = np.concatenate(all_data, axis=0) # (60000, 784)
all_labels = np.concatenate(all_labels, axis=0) # (60000,)

# PCA実行(2次元に圧縮)
pca = PCA(n_components=2)
coords_2d = pca.fit_transform(all_data) # (60000, 2)


## MNIST画像 分布可視化

In [ ]:
def plot_digits_dist_heatmap(coords, labels, target_digits=[0, 1, 9]): 
    plt.figure(figsize=(10, 10))
    
    x_min, x_max = coords[:, 0].min() - 2, coords[:, 0].max() + 2
    y_min, y_max = coords[:, 1].min() - 2, coords[:, 1].max() + 2
    xi, yi = np.mgrid[x_min:x_max:200j, y_min:y_max:200j]
    
    # 0:赤, 1:青, 2:緑, 9:紫
    color_maps = {0: 'Reds', 1: 'Blues', 9: 'Greens'}

    for digit in target_digits:
        mask = (labels == digit)
        if not np.any(mask): continue
        
        data = coords[mask].T
        kde = gaussian_kde(data)
        zi = kde(np.vstack([xi.flatten(), yi.flatten()])).reshape(xi.shape)
        
        zi = np.ma.masked_where(zi < zi.max() * 0.1, zi)
        
        plt.pcolormesh(xi, yi, zi, cmap=color_maps[digit], shading='auto', alpha=0.6)
        
        center = coords[mask].mean(axis=0)
        plt.text(center[0], center[1], f"{digit}", 
                 fontsize=14, fontweight='bold', ha='center', va='center',
                 bbox=dict(facecolor='white', alpha=0.5, edgecolor='none'))

    plt.title("Target Clusters for Flow Matching (0, 1, 9)", fontsize=16)
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.show()

plot_digits_dist_heatmap(coords_2d, all_labels)

## Load Model

In [ ]:
def load_eval_model(model_path, device="cuda"):
    model = MnistFM()
    model = model.to(device)

    checkpoint_path = model_path
    checkpoint = torch.load(checkpoint_path, map_location=device)

    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    return model

model = load_eval_model(model_path, device)

## ノイズ点の移動動画作成

In [ ]:
def create_flow_video(model, target_digit, pca, coords_2d, all_labels, steps=100, n_particles=100):
    ## ==== 背景の準備 ====
    fig, ax = plt.subplots(figsize=(8, 8))
    x_min, x_max = -5, 5
    y_min, y_max = -5, 5
    
    # MNIST 0, 1, 9 の分布を描画
    bg_digits = [0, 1, 9]
    bg_colors = {0: 'Reds', 1: 'Blues', 9: 'Greens'}
    
    xi, yi = np.mgrid[x_min:x_max:150j, y_min:y_max:150j]
    grid_coords = np.vstack([xi.flatten(), yi.flatten()])

    for d in bg_digits:
        mask = (all_labels == d)
        d_coords = coords_2d[mask]
        kde = gaussian_kde(d_coords.T)
        zi = kde(grid_coords).reshape(xi.shape)
        
        # ターゲット以外の分布の濃度調整
        alpha_val = 0.4 if d == target_digit else 0.1
        zi = np.ma.masked_where(zi < zi.max() * 0.1, zi)
        ax.pcolormesh(xi, yi, zi, cmap=bg_colors[d], shading='auto', alpha=alpha_val, zorder=1)
        
        center = d_coords.mean(axis=0)
        ax.text(center[0], center[1], str(d), fontsize=20, weight='bold', alpha=0.3)

    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.axis('off')
    ax.set_title(f"Targeting: {target_digit}", fontsize=14)

    ## ==== ノイズ点の作成 ====
    sc = ax.scatter([], [], c='black', s=30, edgecolors='white', linewidth=0.3, zorder=5, alpha=0.9)

    # 初期ノイズ
    x_t = torch.randn(n_particles, 1, 28, 28).to(device)
    pseudo_prompts = torch.full((n_particles,), target_digit, dtype=torch.long).to(device)
    
    # 軌跡の事前計算
    history_coords = []
    dt = 1.0 / steps
    
    for i in range(steps + 1):
        # 現時点の座標をPCAにより圧縮
        curr_x_flat = torch.clamp(x_t, 0, 1).view(n_particles, -1).cpu().numpy()
        
        p_coords = pca.transform(curr_x_flat)
        history_coords.append(p_coords)
        
        if i < steps:
            t = torch.full((n_particles,), i / steps).to(device)
            with torch.no_grad():
                v = model(pseudo_prompts, t, x_t)
            x_t = x_t + v * dt

    # ==== アニメーションの更新 ====
    def update(frame):
        sc.set_offsets(history_coords[frame])
        return (sc,)

    ani = animation.FuncAnimation(fig, update, frames=len(history_coords), 
                                  interval=50, blit=True)
    
    plt.close()
    return ani

In [ ]:
gen_no = 9 # 生成したい画像の数字

save_path = os.path.join(movie_save_dir, f"floa_animation_digit{gen_no}.gif")
ani = create_flow_video(model, gen_no, pca, coords_2d, all_labels)
ani.save(save_path, writer='pillow', fps=30)
HTML(ani.to_jshtml())